In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

variants = pd.read_csv("/content/drive/MyDrive/cancer_project/training_variants")
text = pd.read_csv("/content/drive/MyDrive/cancer_project/training_text",
                   sep=r"\|\|", engine="python", skiprows=1, names=["ID", "TEXT"])

print(variants.shape, text.shape)
print(variants.head())
print(text.head())

Mounted at /content/drive
(3321, 4) (3321, 2)
   ID    Gene             Variation  Class
0   0  FAM58A  Truncating Mutations      1
1   1     CBL                 W802*      2
2   2     CBL                 Q249E      2
3   3     CBL                 N454D      3
4   4     CBL                 L399V      4
   ID                                               TEXT
0   0  Cyclin-dependent kinases (CDKs) regulate a var...
1   1   Abstract Background  Non-small cell lung canc...
2   2   Abstract Background  Non-small cell lung canc...
3   3  Recent evidence has demonstrated that acquired...
4   4  Oncogenic mutations in the monomeric Casitas B...


In [3]:
df = variants.merge(text, on="ID")
print(df.shape)
print(df["Class"].value_counts().sort_index())
print(df.isnull().sum())

df["TEXT"] = df["TEXT"].fillna("")
df.loc[df["TEXT"] == "", "TEXT"] = df["Gene"] + " " + df["Variation"]

df["words"] = df["TEXT"].apply(lambda x: len(x.split()))
print(df["words"].describe())
print(df["Gene"].nunique(), df["Variation"].nunique())

df["Gene"] = df["Gene"].str.lower().str.strip()
df["Variation"] = df["Variation"].str.lower().str.strip()

df["clean_text"] = (df["TEXT"].str.lower()
                    .str.replace(r"[^a-z0-9 ]", " ", regex=True)
                    .str.replace(r"\s+", " ", regex=True)
                    .str.strip())

print(df[["Gene", "Variation", "Class"]].head())
print(df["clean_text"][0][:300])

(3321, 5)
Class
1    568
2    452
3     89
4    686
5    242
6    275
7    953
8     19
9     37
Name: count, dtype: int64
ID           0
Gene         0
Variation    0
Class        0
TEXT         5
dtype: int64
count     3321.000000
mean      9551.113821
std       7849.193294
min          2.000000
25%       4733.000000
50%       6871.000000
75%      11996.000000
max      76782.000000
Name: words, dtype: float64
264 2996
     Gene             Variation  Class
0  fam58a  truncating mutations      1
1     cbl                 w802*      2
2     cbl                 q249e      2
3     cbl                 n454d      3
4     cbl                 l399v      4
cyclin dependent kinases cdks regulate a variety of fundamental cellular processes cdk10 stands out as one of the last orphan cdks for which no activating cyclin has been identified and no kinase activity revealed previous work has shown that cdk10 silencing increases ets2 v ets erythroblastosis vir


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["Class"])

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=3,
                        max_df=0.9, stop_words="english", sublinear_tf=True)
Xt_train = tfidf.fit_transform(train_df["clean_text"])
Xt_test = tfidf.transform(test_df["clean_text"])

enc = OneHotEncoder(handle_unknown="ignore")
Xg_train = enc.fit_transform(train_df[["Gene", "Variation"]])
Xg_test = enc.transform(test_df[["Gene", "Variation"]])

X_train = hstack([Xt_train, Xg_train]).tocsr()
X_test = hstack([Xt_test, Xg_test]).tocsr()
y_train = train_df["Class"].values
y_test = test_df["Class"].values

print(X_train.shape, X_test.shape)
print(Xt_train.shape, Xg_train.shape)
print("Test variations never seen in training:",
      (~test_df["Variation"].isin(train_df["Variation"])).mean())

(2656, 22644) (665, 22644)
(2656, 20000) (2656, 2644)
Test variations never seen in training: 0.8992481203007519


In [8]:
import numpy as np
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

labels = list(range(1, 10))

shares = np.bincount(y_train, minlength=10)[1:] / len(y_train)
base_probs = np.tile(shares, (len(y_test), 1))

lr = LogisticRegressionCV(Cs=[0.1, 1, 10, 100], cv=3, scoring="neg_log_loss",
                          max_iter=500, n_jobs=-1)
lr.fit(X_train, y_train)
print("Best C:", lr.C_[0])
lr_probs = lr.predict_proba(X_test)

xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.3, tree_method="hist")
xgb.fit(X_train, y_train - 1)
xgb_probs = xgb.predict_proba(X_test)

results = {"Baseline (class shares)": base_probs,
           "Logistic Regression": lr_probs,
           "XGBoost": xgb_probs}

for name, p in results.items():
    ll = log_loss(y_test, p, labels=labels)
    acc = ((p.argmax(axis=1) + 1) == y_test).mean()
    print(name, "| log loss:", round(ll, 4), "| accuracy:", round(acc, 4))

best = min(results, key=lambda n: log_loss(y_test, results[n], labels=labels))
print("Best:", best)
print(pd.crosstab(y_test, results[best].argmax(axis=1) + 1,
                  rownames=["True"], colnames=["Predicted"]))

Best C: 10.0
Baseline (class shares) | log loss: 1.8306 | accuracy: 0.2872
Logistic Regression | log loss: 1.0115 | accuracy: 0.6797
XGBoost | log loss: 1.1431 | accuracy: 0.6256
Best: Logistic Regression
Predicted   1   2  3   4   5   6    7  9
True                                    
1          71   0  0  23  15   3    2  0
2           2  54  0   1   3   0   31  0
3           1   0  5   3   5   0    4  0
4          26   0  2  96   5   3    5  0
5           7   2  0   5  17   2   15  0
6           6   2  0   2   6  29   10  0
7           0  13  1   1   1   0  175  0
8           0   0  0   0   0   0    2  2
9           1   1  0   0   0   0    0  5
